# F2.5 · Validación del dataset resultante

Se comprueba que `data/processed/ordenes.csv` cumple las reglas de calidad antes de usarlo en la Fase 3.

In [1]:
# Configuración común: rutas del proyecto y módulos del pipeline.
import pandas as pd

from pipeline import chilecompra as cc
from pipeline import config, ordenes

pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", "{:,.1f}".format)
print("Raíz del proyecto:", config.PROJECT_ROOT.name)

Raíz del proyecto: 002_sumativa-1-avance-proyecto-f1-y-f2-40


In [2]:
orders = pd.read_csv(config.PROCESSED_DATA_DIR / "ordenes.csv", parse_dates=["FechaEnvio"], dtype={"Codigo": str, "periodo": str, "CodigoProveedor": str, "CodigoUnidadCompra": str})
print(f"{len(orders):,} órdenes, {orders.shape[1]} variables")

10,285 órdenes, 13 variables


## 1. Reglas de validación

`validate_orders` evalúa siete reglas. Si alguna falla, el notebook se detiene y hay que volver a la limpieza.

In [3]:
result = ordenes.validate_orders(orders)
assert result["cumple"].all(), "El dataset no cumple las reglas: revisar el notebook 03"
result

,regla,cumple
0,Hay al menos una orden,True
1,Cada orden aparece una sola vez,True
2,Ningún monto falta ni es negativo,True
3,Todas las órdenes tienen proveedor y unidad de...,True
4,La modalidad solo toma valores conocidos,True
5,El período es uno de los meses del estudio,True
6,La fecha de envío cae en su período,True


## 2. Consistencia antes y después

El pipeline no debe perder órdenes ni dinero respecto de los datos originales.

In [4]:
raw = ordenes.load_orders(config.RAW_DATA_DIR)
raw_amount = ordenes.parse_amount(raw.drop_duplicates("Codigo")["MontoTotalOC_PesosChilenos"]).sum()

assert len(orders) == raw["Codigo"].nunique(), "Se perdieron o duplicaron órdenes"
assert orders["lineas"].sum() == len(raw), "Las líneas no cuadran con el original"
assert abs(orders["MontoTotalOC_PesosChilenos"].sum() - raw_amount) < 1, "El monto total cambió"

pd.DataFrame({"original": [raw["Codigo"].nunique(), len(raw), raw_amount / 1e6], "procesado": [len(orders), orders["lineas"].sum(), orders["MontoTotalOC_PesosChilenos"].sum() / 1e6]}, index=["órdenes", "líneas", "monto (millones de $)"])

,original,procesado
órdenes,"10,285.0","10,285.0"
líneas,"25,880.0","25,880.0"
monto (millones de $),"49,163.3","49,163.3"


## 3. Casos límite y pruebas automáticas

Las funciones del pipeline tienen pruebas en `tests/test_ordenes.py`, que se ejecutan con `pytest` y en cada *push* (GitHub Actions). Cubren casos normales, casos límite y errores. Aquí se muestran dos casos límite sobre datos reales.

In [5]:
# Caso límite 1: una regla debe fallar si se introduce un error a propósito.
broken = pd.concat([orders, orders.iloc[[0]]], ignore_index=True)
failed = ordenes.validate_orders(broken).query("not cumple")["regla"].tolist()
print("Con una orden repetida falla:", failed)
assert failed == ["Cada orden aparece una sola vez"]

# Caso límite 2: un monto que no es número queda como NA en vez de detener el proceso.
print("Montos no numéricos:", ordenes.parse_amount(pd.Series(["1.234,5x", "abc"])).tolist())

Con una orden repetida falla: ['Cada orden aparece una sola vez']
Montos no numéricos: [nan, nan]


In [6]:
# Valores extremos: se revisan, no se eliminan (una compra grande es un dato real).
amount = orders["MontoTotalOC_PesosChilenos"]
print(amount.describe(percentiles=[0.5, 0.9, 0.99]).round(0).to_string())
orders.nlargest(3, "MontoTotalOC_PesosChilenos")[["periodo", "modalidad", "UnidadCompra", "MontoTotalOC_PesosChilenos"]]

count          10,285.0
mean        4,780,098.0
std        27,455,745.0
min             1,071.0
50%           972,230.0
90%         7,654,430.0
99%        64,332,919.0
max     1,601,438,750.0


,periodo,modalidad,UnidadCompra,MontoTotalOC_PesosChilenos
6666,2025-09,Otra,Hospital San Martín de Quillota,"1,601,438,750.0"
4891,2025-09,Otra,SERVICIO DE SALUD VALPARAISO SAN ANTONIO,"1,174,842,822.4"
131,2024-09,Licitación,COMPRAS CONVENIO GORE VALPARAISO,"632,681,319.1"


## 4. Conclusión: dataset apto para la Fase 3

Primera lectura de los indicadores, solo para confirmar que el dataset permite calcularlos. El análisis corresponde a la Fase 3.

In [7]:
summary = orders.groupby(["periodo", "modalidad"]).agg(ordenes=("Codigo", "size"), monto=("MontoTotalOC_PesosChilenos", "sum"))
summary["% de órdenes"] = summary["ordenes"] / summary.groupby("periodo")["ordenes"].transform("sum") * 100
summary["% del monto"] = summary["monto"] / summary.groupby("periodo")["monto"].transform("sum") * 100
summary["monto (millones de $)"] = summary.pop("monto") / 1e6
summary

ordenes  % de órdenes  % del monto  \
periodo modalidad                                            
2024-09 Compra ágil        1768          38.5          6.1   
        Convenio marco      275           6.0          6.1   
        Licitación         1627          35.4         59.5   
        Otra                 79           1.7          2.8   
        Trato directo       849          18.5         25.6   
2025-09 Compra ágil        2423          42.6         11.8   
        Convenio marco      430           7.6          7.5   
        Licitación         1967          34.6         43.3   
        Otra                 93           1.6         16.2   
        Trato directo       774          13.6         21.2   

                        monto (millones de $)  
periodo modalidad                              
2024-09 Compra ágil                   1,280.6  
        Convenio marco                1,283.3  
        Licitación                   12,538.3  
        Otra                            582.2  
        Trato directo                 5,401.4  
2025-09 Compra ágil                   3,303.0  
        Convenio marco                2,111.1  
        Licitación                   12,155.4  
        Otra                          4,554.7  
        Trato directo                 5,953.4

In [8]:
by_supplier = orders.groupby(["periodo", "CodigoProveedor"])["MontoTotalOC_PesosChilenos"].sum()
top10 = by_supplier.groupby("periodo").apply(lambda amounts: amounts.nlargest(10).sum() / amounts.sum() * 100)
top10.rename("% del monto en los 10 mayores proveedores").to_frame()

,% del monto en los 10 mayores proveedores
periodo,
2024-09,22.2
2025-09,26.0


El dataset cumple las siete reglas, conserva todas las órdenes y su monto, y permite calcular los indicadores del proyecto.

**Límites que el informe debe declarar:** son dos meses de una región; los montos son nominales (sin ajuste por inflación); y las licitaciones (`data/raw/lic_*`) quedan disponibles pero todavía no se cruzan con las órdenes.